# StreetForward 演示 Notebook

本 notebook 展示 StreetForward 训练器的基本功能和训练过程。

## 功能概述

### StreetForward
1. Feed-forward 3DGS 训练器，基于代理参数的多视角梯度累积
2. 使用双 NodeState 架构（Background + RigidNodes）存储 Gaussian 参数
3. 通过 MLP 预测偏移量，而不是直接预测参数
4. 使用 Proxy 参数进行渲染，避免二次反传共享图问题
5. 支持多视角监督和梯度回灌机制
6. **新增**：支持测试视角评估（PSNR、SSIM、LPIPS）
7. **新增**：增强的偏移量控制（位置、尺度、旋转、不透明度、SH）
8. **新增**：训练/测试帧分离支持
9. **新增**：动态物体支持（静态背景 + 动态物体联合训练）

### 本 Notebook 包含
1. MultiSceneDataset 数据加载（集成点云生成和动态物体信息）
2. Batch 格式转换（MultiSceneDataset → StreetForward）
3. StreetForwardTrainer 初始化和训练
4. 训练循环演示
5. **测试视角和评估指标演示**（新增）
6. **动态物体支持演示**（新增）
7. 结果可视化

## 使用说明

1. 按顺序执行所有单元格
2. 在"配置准备"部分修改配置文件路径（如果需要）
3. 每个部分可以独立运行和调试
4. 注意内存使用，特别是点云生成和训练部分
5. **测试视角评估**：需要配置 `test_image_stride` 和 `max_test_images` 才能使用
6. **动态物体支持**：需要点云包含 `dynamic` 字段且 `pixel_source` 支持 `instances_pose`

## 更新说明

本 notebook 已更新以使用新的重构实现：
- 使用 `configs/streetforward/multi_scene.yaml` 配置文件
- `MultiSceneDataset` 已集成点云生成功能
- `get_segment_batch()` 自动生成点云并包含在 batch 中
- **新增**：支持测试视角加载和评估（`include_test=True`）
- **新增**：评估指标计算（PSNR、SSIM、LPIPS）
- **新增**：增强的模型配置参数（offset_max, scale_max, omega_max 等）
- **新增**：动态物体信息自动构建（`dynamic_info` 字段）
- **新增**：双 NodeState 架构（Background + RigidNodes）

## 第一部分：环境配置和导入

安装和导入所有必要的依赖包。

In [2]:
# 安装依赖（如果需要）
# !pip install numpy matplotlib open3d omegaconf torch

# 自动加载更改的魔法指令（在开发过程中自动重新加载模块）
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import torch
import types
from omegaconf import OmegaConf
from typing import List, Dict, Optional
import matplotlib.pyplot as plt
import open3d as o3d

# 添加项目路径以导入模块
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

# 导入项目模块
from datasets.multi_scene_dataset import MultiSceneDataset, MultiSceneDatasetScheduler
from models.trainers.streetforward import StreetForwardTrainer
from tools.train_streetforward import convert_batch_to_streetforward_format

# 设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 设置随机种子（可选，用于可重复性）
torch.manual_seed(42)
np.random.seed(42)

print("Environment setup completed!")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


/root/autodl-tmp/conda/envs/drivestudio-new/lib/python3.9/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
Disabling PyTorch because PyTorch >= 2.1 is required but found 2.0.0+cu118
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Using device: cuda
Environment setup completed!


## 第二部分：配置准备

读取配置文件，准备数据配置和 StreetForward 训练器配置。

In [3]:
# 读取配置文件（StreetForward 配置）
config_path = os.path.join(project_root, "configs/streetforward/multi_scene.yaml")
cfg = OmegaConf.load(config_path)

# 提取数据配置
data_cfg = cfg.data

# 提取 MultiSceneDataset 配置（现在在 dataset 下）
dataset_cfg = cfg.dataset


# 显示配置信息
print("Data configuration:")
print(f"  Data root: {data_cfg.data_root}")
print(f"  Dataset type: {data_cfg.dataset}")
print(f"  Train scene IDs: {data_cfg.train_scene_ids}")
print(f"  Eval scene IDs: {data_cfg.eval_scene_ids}")

# 显示测试帧配置（如果存在）
pixel_source_cfg = getattr(data_cfg, "pixel_source", {})
if pixel_source_cfg:
    test_image_stride = pixel_source_cfg.get("test_image_stride", 0)
    max_test_images = pixel_source_cfg.get("max_test_images", 0)
    print(f"  Test image stride: {test_image_stride} (0 = all frames for training)")
    print(f"  Max test images per segment: {max_test_images} (0 = all available)")

print("\nMultiSceneDataset configuration:")
print(f"  Num source keyframes: {dataset_cfg.num_source_keyframes}")
print(f"  Num target keyframes: {dataset_cfg.num_target_keyframes}")
print(f"  Segment overlap ratio: {dataset_cfg.segment_overlap_ratio}")
print(f"  Min keyframes per scene: {dataset_cfg.min_keyframes_per_scene}")
print(f"  Min keyframes per segment: {dataset_cfg.min_keyframes_per_segment}")

# 显示 pointcloud 配置（现在在 dataset.pointcloud 下）
if hasattr(dataset_cfg, 'pointcloud') and dataset_cfg.pointcloud is not None:
    print("\nPoint cloud configuration:")
    print(f"  Type: {dataset_cfg.pointcloud.get('type', 'N/A')}")
    print(f"  Chosen cam IDs: {dataset_cfg.pointcloud.get('chosen_cam_ids', 'N/A')}")
    print(f"  Sparsity: {dataset_cfg.pointcloud.get('sparsity', 'N/A')}")
    print(f"  Filter sky: {dataset_cfg.pointcloud.get('filter_sky', 'N/A')}")
    print(f"  Depth consistency: {dataset_cfg.pointcloud.get('depth_consistency', 'N/A')}")
    print(f"  Use bounding box: {dataset_cfg.pointcloud.get('use_bbx', 'N/A')}")
    print(f"  Downscale: {dataset_cfg.pointcloud.get('downscale', 'N/A')}")

# 准备 fixed_segment_aabb（如果配置了）
fixed_segment_aabb = None
if dataset_cfg.get('fixed_segment_aabb') is not None:
    fixed_segment_aabb = torch.tensor(dataset_cfg.fixed_segment_aabb, dtype=torch.float32)
    print(f"\nUsing fixed segment AABB: {fixed_segment_aabb}")

Data configuration:
  Data root: /root/autodl-tmp/nuScenes/
  Dataset type: nuscenes
  Train scene IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  Eval scene IDs: [10, 11, 12, 13]
  Test image stride: 0 (0 = all frames for training)
  Max test images per segment: 4 (0 = all available)

MultiSceneDataset configuration:
  Num source keyframes: 3
  Num target keyframes: 6
  Segment overlap ratio: 0.2
  Min keyframes per scene: 10
  Min keyframes per segment: 6

Point cloud configuration:
  Type: hybrid
  Chosen cam IDs: N/A
  Sparsity: N/A
  Filter sky: N/A
  Depth consistency: N/A
  Use bounding box: N/A
  Downscale: N/A

Using fixed segment AABB: tensor([[-20.0000, -20.0000, -20.0000],
        [ 20.0000,   4.8000,  70.0000]])


In [4]:
# 创建 StreetForward 训练器配置
# 参考 tests/test_streetforward.py 和模型设计文档

# 从 segment AABB 获取 bbx_min 和 bbx_max（如果有固定AABB）
if fixed_segment_aabb is not None:
    bbx_min = fixed_segment_aabb[0].tolist()
    bbx_max = fixed_segment_aabb[1].tolist()
else:
    # 使用默认值（可以从配置文件中读取）
    bbx_min = [-20.0, -20.0, -20.0]
    bbx_max = [20.0, 4.8, 70.0]

# 从配置文件读取模型参数（如果存在）
model_cfg = cfg.get("model", {})
streetforward_config = cfg
print("StreetForward configuration:")
print(f"  SparseConv output dim: {streetforward_config.model.sparseConv_outdim}")
print(f"  Offset max: {streetforward_config.model.offset_max}")
print(f"  Scale max: {streetforward_config.model.scale_max}")
print(f"  Omega max: {streetforward_config.model.omega_max}")
print(f"  Opacity max: {streetforward_config.model.opacity_max}")
print(f"  SH DC max: {streetforward_config.model.sh_dc_max}")
print(f"  SH rest max: {streetforward_config.model.sh_rest_max}")
print(f"  Eta (step size factors): means={streetforward_config.model.eta_means}, scales={streetforward_config.model.eta_scales}, opacity={streetforward_config.model.eta_opacity}")
print(f"  SH degree: {streetforward_config.model.sh_degree}")
print(f"  Voxel size: {streetforward_config.model.voxel_size}")
print(f"  Max iterations: {streetforward_config.model.max_iterations}")
print(f"  Bounding box min: {streetforward_config.model.bbx_min}")
print(f"  Bounding box max: {streetforward_config.model.bbx_max}")
print(f"  Learning rate: {streetforward_config.optimizer.lr}")

StreetForward configuration:
  SparseConv output dim: 32
  Offset max: 0.1
  Scale max: 0.1
  Omega max: 0.1
  Opacity max: 0.1
  SH DC max: 0.1
  SH rest max: 0.05
  Eta (step size factors): means=1.0, scales=1.0, opacity=1.0
  SH degree: 1
  Voxel size: 0.4
  Max iterations: 1
  Bounding box min: [-20.0, -20.0, -20.0]
  Bounding box max: [20.0, 4.8, 70.0]
  Learning rate: 0.001


In [5]:
# 准备 pointcloud 配置（从 dataset_cfg.pointcloud 读取）
pointcloud_config = None
if hasattr(dataset_cfg, 'pointcloud') and dataset_cfg.pointcloud is not None:
    pointcloud_config = dict(dataset_cfg.pointcloud)
    print("Pointcloud configuration prepared from dataset_cfg.pointcloud")
else:
    print("Warning: No pointcloud config found. Pointcloud will not be generated automatically.")

# 创建 MultiSceneDataset 实例（现在会自动创建 pointcloud_generator）
dataset = MultiSceneDataset(
    data_cfg=data_cfg,
    train_scene_ids=data_cfg.train_scene_ids,
    eval_scene_ids=data_cfg.eval_scene_ids,
    num_source_keyframes=dataset_cfg.num_source_keyframes,
    num_target_keyframes=dataset_cfg.num_target_keyframes,
    segment_overlap_ratio=dataset_cfg.segment_overlap_ratio,
    keyframe_split_config=dict(dataset_cfg.keyframe_split_config) if hasattr(dataset_cfg, 'keyframe_split_config') else None,
    min_keyframes_per_scene=dataset_cfg.min_keyframes_per_scene,
    min_keyframes_per_segment=dataset_cfg.min_keyframes_per_segment,
    device=device,
    preload_scene_count=1,  # 预加载2个场景（减少内存占用）
    fixed_segment_aabb=fixed_segment_aabb,
    pointcloud_config=pointcloud_config,  # 传入 pointcloud 配置（注意参数名是 pointcloud_config）
)

print("MultiSceneDataset created successfully!")
if dataset.pointcloud_generator is not None:
    print("  Pointcloud generator initialized automatically")
else:
    print("  Warning: No pointcloud generator initialized")

StreetForward requires a single source keyframe; overriding num_source_keyframes=3 to 1


Pointcloud configuration prepared from dataset_cfg.pointcloud
MultiSceneDataset created successfully!
  Pointcloud generator initialized automatically


In [6]:
# 初始化数据集（可选，会在第一次使用时自动初始化）
dataset.initialize()

# 获取当前场景ID
current_scene_id = dataset.get_current_scene_id()
print(f"Current training scene ID: {current_scene_id}")

# 获取场景信息（如果场景已加载）
if current_scene_id is not None:
    scene_info = dataset.get_scene(current_scene_id)
    if scene_info:
        print(f"\nScene {current_scene_id} information:")
        print(f"  Number of segments: {len(scene_info['segments'])}")
        print(f"  Number of frames: {scene_info['num_frames']}")
        print(f"  Number of cameras: {scene_info['num_cams']}")
        print(f"  Number of keyframe segments: {len(scene_info['keyframe_segments'])}")
        
        # 显示训练帧和测试帧信息（如果可用）
        if 'train_frame_indices' in scene_info and 'test_frame_indices' in scene_info:
            train_frames = scene_info['train_frame_indices']
            test_frames = scene_info['test_frame_indices']
            print(f"  Training frames: {len(train_frames)} frames")
            print(f"  Test frames: {len(test_frames)} frames")
            if len(test_frames) > 0:
                print(f"    Test frame indices (first 10): {test_frames[:10]}")
        
        # 显示每个段的信息
        for i, segment in enumerate(scene_info['segments']):
            print(f"\n  Segment {i}:")
            print(f"    Keyframe indices: {segment['keyframe_indices']}")
            print(f"    Number of frames: {len(segment['frame_indices'])}")
            print(f"    AABB shape: {segment['aabb'].shape}")
            # 显示段内的测试帧（如果可用）
            if 'test_frame_indices' in segment:
                test_frames_in_seg = segment['test_frame_indices']
                print(f"    Test frames in segment: {len(test_frames_in_seg)}")
                if len(test_frames_in_seg) > 0:
                    print(f"      Test frame indices: {test_frames_in_seg[:10]}{'...' if len(test_frames_in_seg) > 10 else ''}")

Loading lidar: 100%|██████████| 196/196 [00:01<00:00, 174.79it/s]
Projecting lidar pts on images for camera CAM_FRONT: 100%|██████████| 196/196 [00:04<00:00, 41.97it/s]
Projecting lidar pts on images for camera CAM_FRONT_LEFT: 100%|██████████| 196/196 [00:04<00:00, 39.47it/s]
Projecting lidar pts on images for camera CAM_FRONT_RIGHT: 100%|██████████| 196/196 [00:04<00:00, 41.19it/s]
Scene 2 is not suitable for training (insufficient keyframes), skipping...
Loading lidar: 100%|██████████| 196/196 [00:01<00:00, 157.34it/s]
Projecting lidar pts on images for camera CAM_FRONT: 100%|██████████| 196/196 [00:05<00:00, 34.05it/s]
Projecting lidar pts on images for camera CAM_FRONT_LEFT: 100%|██████████| 196/196 [00:05<00:00, 32.93it/s]
Projecting lidar pts on images for camera CAM_FRONT_RIGHT: 100%|██████████| 196/196 [00:05<00:00, 35.02it/s]
Loading lidar: 100%|██████████| 191/191 [00:01<00:00, 160.78it/s]
Projecting lidar pts on images for camera CAM_FRONT: 100%|██████████| 191/191 [00:05<00

Current training scene ID: 1

Scene 1 information:
  Number of segments: 2
  Number of frames: 196
  Number of cameras: 3
  Number of keyframe segments: 37
  Training frames: 196 frames
  Test frames: 196 frames
    Test frame indices (first 10): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

  Segment 0:
    Keyframe indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
    Number of frames: 112
    AABB shape: torch.Size([2, 3])
    Test frames in segment: 112
      Test frame indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]...

  Segment 1:
    Keyframe indices: [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
    Number of frames: 79
    AABB shape: torch.Size([2, 3])
    Test frames in segment: 79
      Test frame indices: [99, 100, 101, 102, 103, 104, 105, 106, 107, 108]...


## 第四部分：点云生成和 Web 可视化演示

本部分演示如何使用 MultiSceneDataset 的点云生成器生成点云，并使用 Web 可视化工具在浏览器中查看点云。

### 功能说明

1. **点云生成**：使用 `dataset.pointcloud_generator` 为指定场景和段生成 RGB 点云
2. **点云格式**：返回字典包含：
   - `background`: [N, 6] 背景点云（世界坐标系，xyz + rgb）
   - `dynamic`: Dict[int, [M, 6]] 动态对象点云（局部坐标系，xyz + rgb）
   - `instance_mapping`: 实例ID映射
   - `metadata`: 元数据信息
3. **Web 可视化**：使用 `PLYViewer` 在浏览器中可视化点云（基于 viser/nerfstudio viewer）

### 参考文档

- 点云生成器文档：`docs/pointcloud_generators/PointCloud_Generators.md`
- Web 可视化工具：`tools/plyviewer.py`

In [7]:
# 检查点云生成器是否可用
if dataset.pointcloud_generator is None:
    print("警告：点云生成器未初始化。请检查配置文件中的 pointcloud 配置。")
    print("点云生成器需要在创建 MultiSceneDataset 时通过 pointcloud_config 参数传入配置。")
else:
    print("点云生成器已初始化！")
    print(f"  类型: {type(dataset.pointcloud_generator).__name__}")
    
    # 显示点云生成器配置
    if hasattr(dataset.pointcloud_generator, 'chosen_cam_ids'):
        print(f"  使用的相机ID: {dataset.pointcloud_generator.chosen_cam_ids}")
    print(f"  稀疏度: {dataset.pointcloud_generator.sparsity}")
    print(f"  过滤天空: {dataset.pointcloud_generator.filter_sky}")
    print(f"  深度一致性检查: {dataset.pointcloud_generator.depth_consistency}")
    print(f"  使用边界框: {dataset.pointcloud_generator.use_bbx}")
    print(f"  下采样比例: {dataset.pointcloud_generator.downscale}")

点云生成器已初始化！
  类型: HybridRGBPointCloudGenerator
  稀疏度: full
  过滤天空: False
  深度一致性检查: False
  使用边界框: True
  下采样比例: 1


In [22]:
# 生成点云演示
# 选择一个场景和段来生成点云

if dataset.pointcloud_generator is not None:
    # 获取当前场景ID
    scene_id = dataset.get_current_scene_id()
    if scene_id is not None:
        scene_info = dataset.get_scene(scene_id)
        if scene_info and len(scene_info['segments']) > 0:
            # 选择第一个段进行演示
            segment_id = 0
            segment = scene_info['segments'][segment_id]
            
            print(f"生成点云：Scene {scene_id}, Segment {segment_id}")
            print(f"  段包含 {len(segment['frame_indices'])} 帧")
            
            # 获取段的AABB（如果可用）
            if 'aabb' in segment:
                segment_aabb = segment['aabb']
                print(f"  段AABB: min={segment_aabb[0].cpu().numpy()}, max={segment_aabb[1].cpu().numpy()}")
            
            print("\n开始生成点云（这可能需要一些时间）...")
            
            try:
                # 生成点云
                pointcloud = dataset.pointcloud_generator.generate_pointcloud(
                    dataset=dataset,
                    scene_id=scene_id,
                    segment_id=segment_id,
                )
                
                # 显示点云统计信息
                print("\n点云生成成功！")
                print("\n点云统计信息：")
                
                background = pointcloud.get("background", np.zeros((0, 6), dtype=np.float32))
                dynamic = pointcloud.get("dynamic", {})
                instance_mapping = pointcloud.get("instance_mapping", {})
                metadata = pointcloud.get("metadata", {})
                
                # 辅助函数：将四元数转换为旋转矩阵（与 streetforward.py 中的实现一致）
                def quat_to_rotmat(q):
                    """将四元数（wxyz 格式）转换为旋转矩阵。"""
                    q = q / (np.linalg.norm(q) + 1e-8)  # 归一化
                    w, x, y, z = q
                    rot = np.array([
                        [1.0 - 2.0 * (y*y + z*z), 2.0 * (x*y - w*z), 2.0 * (x*z + w*y)],
                        [2.0 * (x*y + w*z), 1.0 - 2.0 * (x*x + z*z), 2.0 * (y*z - w*x)],
                        [2.0 * (x*z - w*y), 2.0 * (y*z + w*x), 1.0 - 2.0 * (x*x + y*y)]
                    ])
                    return rot
                
                # 辅助函数：将局部坐标点云变换到世界坐标
                def transform_local_to_world(points_local, quat, trans):
                    """将局部坐标的点云变换到世界坐标。"""
                    rot = quat_to_rotmat(quat)
                    points_world = (rot @ points_local.T).T + trans
                    return points_world
                
                # 获取动态物体位姿信息并应用变换
                if len(dynamic) > 0:
                    scene_dataset = scene_info['dataset']
                    frame_indices = segment['frame_indices']
                    dynamic_info = dataset._build_dynamic_info(
                        scene_dataset=scene_dataset,
                        frame_indices=frame_indices,
                        instance_mapping=instance_mapping,
                    )
                    
                    # 如果存在动态物体和位姿信息，应用位姿变换
                    if dynamic_info is not None:
                        print("\n应用动态物体位姿变换...")
                        # 选择第一帧的位姿进行变换（用于演示）
                        first_frame_idx = frame_indices[0] if frame_indices else None
                        
                        if first_frame_idx is not None and first_frame_idx in dynamic_info:
                            frame_info = dynamic_info[first_frame_idx]
                            instances = frame_info.get("instances", {})
                            
                            # 创建变换后的动态点云字典
                            dynamic_transformed = {}
                            
                            for intid, pts_local in dynamic.items():
                                intid_key = int(intid)
                                if intid_key in instances:
                                    instance_pose = instances[intid_key]
                                    quat = np.array(instance_pose["quat"])  # [w, x, y, z]
                                    trans = np.array(instance_pose["trans"])  # [x, y, z]
                                    
                                    # 提取局部坐标点
                                    points_local_coords = pts_local[:, :3]  # [N, 3]
                                    colors = pts_local[:, 3:]  # [N, 3]
                                    
                                    # 应用位姿变换到世界坐标
                                    points_world = transform_local_to_world(points_local_coords, quat, trans)
                                    
                                    # 合并坐标和颜色
                                    dynamic_transformed[intid] = np.concatenate([points_world, colors], axis=1)
                                else:
                                    # 如果没有位姿信息，保持原样（局部坐标）
                                    dynamic_transformed[intid] = pts_local
                            
                            # 更新点云中的动态物体
                            pointcloud["dynamic"] = dynamic_transformed
                            dynamic = dynamic_transformed
                            print(f"  已对 {len(dynamic_transformed)} 个动态物体应用位姿变换")
                        else:
                            print(f"  警告：无法找到帧 {first_frame_idx} 的位姿信息，跳过变换")
                    else:
                        print("\n  警告：无法获取动态物体位姿信息，动态物体保持局部坐标")
                
                # 显示点云统计信息
                num_background_points = len(background)
                num_dynamic_points = sum(len(pts) for pts in dynamic.values())
                total_points = num_background_points + num_dynamic_points
                
                print(f"  总点数: {total_points:,}")
                print(f"    背景点数: {num_background_points:,}")
                print(f"    动态对象点数: {num_dynamic_points:,}")
                print(f"    动态对象数量: {len(dynamic)}")
                
                if num_background_points > 0:
                    background_points = background[:, :3]
                    background_colors = background[:, 3:]
                    print(f"\n  背景点云：")
                    print(f"    坐标范围 X: [{background_points[:, 0].min():.2f}, {background_points[:, 0].max():.2f}]")
                    print(f"    坐标范围 Y: [{background_points[:, 1].min():.2f}, {background_points[:, 1].max():.2f}]")
                    print(f"    坐标范围 Z: [{background_points[:, 2].min():.2f}, {background_points[:, 2].max():.2f}]")
                    print(f"    颜色范围: [{background_colors.min():.0f}, {background_colors.max():.0f}]")
                
                if len(dynamic) > 0:
                    print(f"\n  动态对象点云：")
                    for intid, pts in dynamic.items():
                        print(f"    对象 {intid}: {len(pts):,} 点")
                
                if metadata:
                    print(f"\n  元数据：")
                    for key, value in metadata.items():
                        print(f"    {key}: {value}")
                
                # 保存点云到变量供后续可视化使用
                demo_pointcloud = pointcloud
                demo_scene_id = scene_id
                demo_segment_id = segment_id
                
            except Exception as e:
                print(f"\n点云生成失败: {e}")
                import traceback
                traceback.print_exc()
                demo_pointcloud = None
        else:
            print(f"场景 {scene_id} 没有可用的段")
            demo_pointcloud = None
    else:
        print("没有当前场景可用，请先初始化数据集")
        demo_pointcloud = None
else:
    print("点云生成器未初始化，跳过点云生成演示")
    demo_pointcloud = None

生成点云：Scene 1, Segment 0
  段包含 112 帧
  段AABB: min=[-20. -20. -20.], max=[20.   4.8 70. ]

开始生成点云（这可能需要一些时间）...

点云生成成功！

点云统计信息：

应用动态物体位姿变换...
  已对 12 个动态物体应用位姿变换
  总点数: 566,642
    背景点数: 500,000
    动态对象点数: 66,642
    动态对象数量: 12

  背景点云：
    坐标范围 X: [-20.00, 20.00]
    坐标范围 Y: [-19.92, 3.73]
    坐标范围 Z: [-7.15, 70.00]
    颜色范围: [0, 255]

  动态对象点云：
    对象 0: 240 点
    对象 2: 872 点
    对象 4: 133 点
    对象 6: 1,449 点
    对象 7: 17,848 点
    对象 9: 160 点
    对象 10: 542 点
    对象 11: 574 点
    对象 14: 33,258 点
    对象 21: 10,435 点
    对象 16: 1,088 点
    对象 22: 43 点

  元数据：
    type: hybrid
    lidar_count: 205097
    monocular_count: 4992337
    fused_background_count: 500000
    dynamic_count: 66642
    fusion_strategy: merge
    max_points: 500000
    dynamic_source: lidar_only
    lidar_frames_used: 82
    monocular_frames_used: 112


In [ ]:
# Web 可视化点云（使用 PLYViewer）
# 注意：这会在浏览器中打开一个 Web 界面来查看点云

if 'demo_pointcloud' in locals() and demo_pointcloud is not None:
    from tools.plyviewer import PLYViewer
    
    
    # 创建 viewer 实例
    viewer = PLYViewer(
        host="0.0.0.0",  # 监听所有接口
        port=7007,  # 默认端口，如果被占用会自动选择其他端口
        point_size=0.01,  # 点的大小
        point_shape="circle",  # 点的形状
        auto_fallback=True,  # 如果端口被占用，自动查找可用端口
    )
    
    # 启动 viewer
    viewer.start_viewer()
    
    # 添加背景点云
    background = demo_pointcloud.get("background", np.zeros((0, 6), dtype=np.float32))
    if len(background) > 0:
        background_points = background[:, :3]  # [N, 3]
        background_colors = background[:, 3:]  # [N, 3], 范围 [0, 255]
        
        # 下采样以加快可视化（如果点太多）
        max_points_for_viz = 100000  # 最多显示10万个点
        if len(background_points) > max_points_for_viz:
            indices = np.random.choice(len(background_points), max_points_for_viz, replace=False)
            background_points = background_points[indices]
            background_colors = background_colors[indices]
        viewer.add_point_cloud(
            points=background_points,
            colors=background_colors,
            name="/background",
            visible=True,
        )
        print(f"  ✓ 已添加背景点云: {len(background_points):,} 点")
    
    # 添加动态对象点云
    dynamic = demo_pointcloud.get("dynamic", {})
    if len(dynamic) > 0:
        for intid, pts in dynamic.items():
            if len(pts) > 0:
                dynamic_points = pts[:, :3]  # [M, 3]
                dynamic_colors = pts[:, 3:]  # [M, 3], 范围 [0, 255]
                
                # 下采样动态对象点云
                max_dynamic_points = 10000
                if len(dynamic_points) > max_dynamic_points:
                    indices = np.random.choice(len(dynamic_points), max_dynamic_points, replace=False)
                    dynamic_points = dynamic_points[indices]
                    dynamic_colors = dynamic_colors[indices]
                
                viewer.add_point_cloud(
                    points=dynamic_points,
                    colors=dynamic_colors,
                    name=f"/dynamic/object_{intid}",
                    visible=True,
                )
    print(f"\n{viewer.viewer_info}")
    
    # 保存 viewer 实例供后续使用
    demo_viewer = viewer
else:
    print("没有可用的点云数据，跳过可视化")
    print("请先执行点云生成 cell")

[16:30:05] Port 7007 is already in use. Attempting to free the port (only viewer processes)...     ]8;id=43217;file:///root/drivestudio-coding/tools/plyviewer.py\plyviewer.py]8;;\:]8;id=639846;file:///root/drivestudio-coding/tools/plyviewer.py#264\264]8;;\

[16:30:06] Could not check processes on port 7007: [Errno 2] No such file or directory: 'lsof'     ]8;id=682758;file:///root/drivestudio-coding/tools/plyviewer.py\plyviewer.py]8;;\:]8;id=749585;file:///root/drivestudio-coding/tools/plyviewer.py#247\247]8;;\

           Could not free port 7007 (may be used by other processes). Automatically finding an     ]8;id=124096;file:///root/drivestudio-coding/tools/plyviewer.py\plyviewer.py]8;;\:]8;id=346903;file:///root/drivestudio-coding/tools/plyviewer.py#279\279]8;;\
           available port...                                                                                       

           Using port 35181 instead. Access viewer at: http://localhost:35181                      ]8;id=366438;file:///root/drivestudio-coding/tools/plyviewer.py\plyviewer.py]8;;\:]8;id=295837;file:///root/drivestudio-coding/tools/plyviewer.py#284\284]8;;\

           Starting viewer server on 0.0.0.0:35181                                                 ]8;id=591408;file:///root/drivestudio-coding/tools/plyviewer.py\plyviewer.py]8;;\:]8;id=343555;file:///root/drivestudio-coding/tools/plyviewer.py#370\370]8;;\

...

(viser) Connection opened (1, 1 total), 55 persistent messages

(viser) Connection closed (1, 0 total)

(viser) Connection closed (0, 0 total)

(viser) Connection closed (0, 0 total)

(viser) Connection opened (1, 1 total), 55 persistent messages

## 第六部分：StreetForwardTrainer 初始化

创建 StreetForwardTrainer 实例并展示模型结构。

In [10]:
# 创建 StreetForwardTrainer 实例
trainer = StreetForwardTrainer(
    config=streetforward_config,
    device=device,
)

print("StreetForwardTrainer created successfully!")
print(f"\nModel structure:")
print(f"  SparseConv: {type(trainer.sparse_conv).__name__}")
print(f"  MLP Offset Position: {trainer.mlp_offset_pos}")
print(f"  MLP Conv (scales + axis-angle): {trainer.mlp_conv}")
print(f"    Note: Now outputs 6 dims (3 for scales + 3 for axis-angle) instead of 7")
print(f"  MLP Opacity: {trainer.mlp_opacity}")
print(f"  Gaussian Decoder (SH): {trainer.gaussion_decoder}")

# 计算参数量
total_params = sum(p.numel() for p in trainer.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params:,}")

# 显示优化器配置
print(f"\nOptimizer:")
print(f"  Type: {type(trainer.optimizer).__name__}")
print(f"  Learning rate: {trainer.optimizer.param_groups[0]['lr']}")
print(f"  Eps: {trainer.optimizer.param_groups[0]['eps']}")
print(f"  Weight decay: {trainer.optimizer.param_groups[0]['weight_decay']}")

# 检查节点状态字典（初始为空）
print(f"\nNode states (initial): {len(trainer.node_states)} nodes")

StreetForwardTrainer created successfully!

Model structure:
  SparseConv: SparseCostRegNet
  MLP Offset Position: Sequential(
  (0): Linear(in_features=48, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=3, bias=True)
)
  MLP Conv (scales + axis-angle): Sequential(
  (0): Linear(in_features=48, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=6, bias=True)
)
    Note: Now outputs 6 dims (3 for scales + 3 for axis-angle) instead of 7
  MLP Opacity: Sequential(
  (0): Linear(in_features=48, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)
  Gaussian Decoder (SH): Sequential(
  (0): Linear(in_features=48, out_features=64, bias=True)
  (1): ReLU()
  (2):

## 第八部分：训练循环演示

使用调度器进行简化的训练循环，展示损失变化。

In [11]:
# 创建调度器
# 注意：可以设置 include_test=True 来在训练时包含测试视角（用于评估）
scheduler = dataset.create_scheduler(
    batches_per_segment=5,  # 每个段遍历5次（演示用，实际训练可以更多）
    segment_order="random",
    scene_order="random",
    shuffle_segments=True,
    preload_next_scene=True,
    include_test=True,  # 设置为 True 可以在训练时包含测试视角
)

print("Scheduler created successfully!")
print(f"  Batches per segment: 5")
print(f"  Segment order: random")
print(f"  Scene order: random")
print(f"  Include test views: False (set to True to include test views in batches)")

Scheduler created successfully!
  Batches per segment: 5
  Segment order: random
  Scene order: random
  Include test views: False (set to True to include test views in batches)


In [12]:
# 简化的训练循环（少量迭代，仅演示）
num_iterations = 3  # 演示用，只运行3次迭代

losses = []
scene_ids_list = []
segment_ids_list = []

try:
    for iteration in range(num_iterations):
        # 获取下一个 batch
        try:
            multi_scene_batch = scheduler.next_batch()
        except StopIteration:
            print("All scenes processed. Resetting scheduler...")
            scheduler.reset()
            multi_scene_batch = scheduler.next_batch()
        
        scene_id = multi_scene_batch['scene_id'].item() if isinstance(multi_scene_batch['scene_id'], torch.Tensor) else multi_scene_batch['scene_id']
        segment_id = multi_scene_batch['segment_id']
        
        # 获取当前状态信息
        info = scheduler.get_current_info()
        
        print(f"\nIteration {iteration + 1}/{num_iterations}:")
        print(f"  Scene ID: {scene_id}, Segment ID: {segment_id}")
        print(f"  Batch count: {info['batch_count']}/{info['batches_per_segment']}")
        
        # 注意：点云已经在 batch 中（get_segment_batch 自动生成）
        # 检查点云是否存在且不为空
        if 'pointcloud' not in multi_scene_batch:
            print("  Warning: No pointcloud in batch. Skipping this iteration.")
            continue
        
        # 检查点云是否为空
        pointcloud = multi_scene_batch['pointcloud']
        if isinstance(pointcloud, dict):
            background = pointcloud.get("background", np.zeros((0, 6), dtype=np.float32))
            if len(background) == 0:
                print(f"  Warning: Empty pointcloud for scene {scene_id}, segment {segment_id}. Skipping this iteration.")
                print(f"    This may happen if all points were filtered out (depth consistency, sky filtering, bbox cropping, etc.)")
                continue
            print(f"  Pointcloud background shape: {background.shape}")
        else:
            # 对于非字典格式的点云，也检查是否为空
            points = np.asarray(pointcloud.points) if hasattr(pointcloud, 'points') else None
            if points is None or len(points) == 0:
                print(f"  Warning: Empty pointcloud for scene {scene_id}, segment {segment_id}. Skipping this iteration.")
                continue
        
        # 转换 batch 格式（点云已包含在 batch 中）
        streetforward_batch = convert_batch_to_streetforward_format(
            batch=multi_scene_batch,
            device=device,
        )
        
        # 运行训练迭代（这次会更新状态）
        outputs = trainer.train_iter(
            batch=streetforward_batch,
            apply_update=True,  # 更新优化器
            update_state=True,  # 更新 node_state
        )
        
        loss = outputs['total_loss'].item()
        losses.append(loss)
        scene_ids_list.append(scene_id)
        segment_ids_list.append(segment_id)
        
        print(f"  Loss: {loss:.6f}")
        print(f"  Number of target views: {len(streetforward_batch['target_views'])}")
        # 检查是否有动态物体
        if 'dynamic_info' in streetforward_batch and streetforward_batch['dynamic_info'] is not None:
            print(f"  Dynamic info: {len(streetforward_batch['dynamic_info'])} frames")
        if hasattr(trainer, 'node_states_rigid'):
            key = (scene_id, segment_id)
            if key in trainer.node_states_rigid and trainer.node_states_rigid[key] is not None:
                num_instances = len(trainer.node_states_rigid[key].instance_ids) if trainer.node_states_rigid[key].instance_ids else trainer.node_states_rigid[key].instances_quats.shape[1]
                print(f"  Dynamic objects: {num_instances} instances")
        
        # 释放内存（可选）
        del multi_scene_batch, streetforward_batch, outputs
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

except KeyboardInterrupt:
    print("\nTraining interrupted by user.")
except Exception as e:
    print(f"\nError during training: {e}")
    import traceback
    traceback.print_exc()
finally:
    # 清理调度器
    scheduler.shutdown()

print("\n" + "-" * 80)
print("Training loop completed!")
print(f"  Total iterations: {len(losses)}")
if losses:
    print(f"  Final loss: {losses[-1]:.6f}")
    print(f"  Average loss: {np.mean(losses):.6f}")
    print(f"  Min loss: {np.min(losses):.6f}")
    print(f"  Max loss: {np.max(losses):.6f}")
else:
    print("  Final loss: N/A")
    print("  Average loss: N/A")
    print("  Min loss: N/A")
    print("  Max loss: N/A")

(viser) Connection opened (0, 1 total), 55 persistent messages


Iteration 1/3:
  Scene ID: 1, Segment ID: 1
  Batch count: 1/5
  Pointcloud background shape: (500000, 6)


/root/drivestudio-coding/models/trainers/streetforward.py:3315: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at aten/src/ATen/core/TensorBody.h:486.)
  if key in self._last_offsets_bg and self._last_offsets_bg[key].grad is not None:


  Loss: 0.162942
  Number of target views: 18
  Dynamic info: 79 frames

Iteration 2/3:
  Scene ID: 1, Segment ID: 1
  Batch count: 2/5
  Pointcloud background shape: (500000, 6)
  Loss: 0.169654
  Number of target views: 18
  Dynamic info: 79 frames

Iteration 3/3:
  Scene ID: 1, Segment ID: 1
  Batch count: 3/5
  Pointcloud background shape: (500000, 6)
  Loss: 0.139227
  Number of target views: 18
  Dynamic info: 79 frames

--------------------------------------------------------------------------------
Training loop completed!
  Total iterations: 3
  Final loss: 0.139227
  Average loss: 0.157274
  Min loss: 0.139227
  Max loss: 0.169654


## 第十一部分：gaussians_all 渲染检查演示

本部分演示如何通过渲染检查 `gaussians_all` 是否正确。使用 `_prepare_all_gaussians` 方法合并三类点（背景、动态、远景），然后使用 nerfview 进行渲染可视化。

### 功能说明

1. **获取 NodeStates**：从 `_get_or_init_node_states` 获取 `node_state_bg`, `node_state_rigid`, `node_state_distant`
2. **合并高斯参数**：使用 `_prepare_all_gaussians` 合并所有高斯参数
3. **渲染可视化**：使用 nerfview 创建交互式渲染器，检查合并后的高斯是否正确

In [15]:
# 导入必要的模块
from typing import Tuple
from models.gaussians.basics import dataclass_camera, dataclass_gs
from gsplat.rendering import rasterization
import viser
import nerfview

print("Gaussians All Rendering Check Demo")
print("=" * 80)
print("Note: This demo shows all Gaussians (including invisible rigid points).")
print("      For training, rigid points are filtered by visibility masks.")
print("      See docs/analysis/rigid_visibility_filtering_minimal_change_proposal.md for details.")

# 检查 trainer 是否已初始化
if 'trainer' not in locals() or trainer is None:
    print("Error: Trainer not initialized. Please run the trainer initialization cell first.")
else:
    # 获取一个 batch（如果还没有的话）
    if 'streetforward_batch' not in locals() or streetforward_batch is None:
        # 尝试从 dataset 获取 batch
        scene_id = dataset.get_current_scene_id() if 'dataset' in locals() else None
        if scene_id is not None:
            multi_scene_batch = dataset.get_segment_batch(
                scene_id=scene_id,
                segment_id=0,
                include_test=False,
            )
            streetforward_batch = convert_batch_to_streetforward_format(
                batch=multi_scene_batch,
                device=device,
            )
        else:
            print("Error: No dataset available. Please initialize dataset first.")
            streetforward_batch = None
    
    if streetforward_batch is not None:
        # 获取 node states
        print("\n1. Getting node states from _get_or_init_node_states...")
        key, node_state_bg, node_state_rigid, node_state_distant = trainer._get_or_init_node_states(streetforward_batch)
        
        print(f"   Key: {key}")
        print(f"   Background NodeState: {node_state_bg.means.shape[0]} Gaussians")
        if node_state_rigid is not None:
            print(f"   Rigid NodeState: {node_state_rigid.means.shape[0]} Gaussians")
            # 显示 rigid 点的可见性信息（如果可用）
            if hasattr(node_state_rigid, 'instances_fv') and node_state_rigid.frame_ids:
                num_frames = len(node_state_rigid.frame_ids)
                num_instances = node_state_rigid.instances_fv.shape[1] if len(node_state_rigid.instances_fv.shape) > 1 else 0
                print(f"     - Rigid instances: {num_instances} instances across {num_frames} frames")
                if num_frames > 0:
                    # 计算第一个帧的可见点数量
                    first_frame_idx = node_state_rigid.frame_ids[0]
                    resolved_idx = trainer._resolve_rigid_frame_idx(node_state_rigid, first_frame_idx)
                    if resolved_idx is not None:
                        visibility = node_state_rigid.instances_fv[resolved_idx]
                        point_visibility = visibility[node_state_rigid.point_ids[..., 0]].bool()
                        num_visible = point_visibility.sum().item()
                        print(f"     - Visible in first frame ({first_frame_idx}): {num_visible}/{len(point_visibility)} points")
        else:
            print(f"   Rigid NodeState: None")
        if node_state_distant is not None:
            print(f"   Distant NodeState: {node_state_distant.means.shape[0]} Gaussians")
        else:
            print(f"   Distant NodeState: None")
        
        # 获取 source_frame_idx（必须是有效的 frame ID，不是索引）
        # 优先从 node_state_rigid.frame_ids 获取（如果有 rigid node state）
        # 否则从 batch 的 source_frame_indices 获取
        source_frame_idx = None
        if node_state_rigid is not None and node_state_rigid.frame_ids:
            # 使用第一个有效的 frame ID
            source_frame_idx = node_state_rigid.frame_ids[0]
            print(f"   Using first frame ID from node_state_rigid.frame_ids: {source_frame_idx}")
        elif 'source_frame_indices' in streetforward_batch and len(streetforward_batch['source_frame_indices']) > 0:
            source_frame_idx = streetforward_batch['source_frame_indices'][0].item() if isinstance(streetforward_batch['source_frame_indices'][0], torch.Tensor) else streetforward_batch['source_frame_indices'][0]
            print(f"   Using first frame ID from batch.source_frame_indices: {source_frame_idx}")
        else:
            # 如果没有 rigid node state，可以使用任意值（不会被使用）
            source_frame_idx = 0
            print(f"   No rigid node state, using default source_frame_idx: {source_frame_idx}")
        
        # 使用 _prepare_all_gaussians 合并所有高斯
        # 注意：这个方法会显示所有点（包括不可见的 rigid 点）
        # 在实际训练中，rigid 点会根据可见性 mask 进行过滤
        print(f"\n2. Preparing all Gaussians using _prepare_all_gaussians (source_frame_idx={source_frame_idx})...")
        print(f"   Note: This includes ALL rigid points for visualization.")
        print(f"         In training, only visible rigid points are used (see visibility filtering proposal).")
        gaussians_all, num_bg, num_rigid, num_distant = trainer._prepare_all_gaussians(
            node_state_bg=node_state_bg,
            node_state_rigid=node_state_rigid,
            node_state_distant=node_state_distant,
            source_frame_idx=source_frame_idx,
        )
        
        print(f"   Merged Gaussians:")
        print(f"     Total: {gaussians_all['means'].shape[0]} Gaussians")
        print(f"     Background: {num_bg}")
        print(f"     Rigid: {num_rigid}")
        print(f"     Distant: {num_distant}")
        print(f"   Means shape: {gaussians_all['means'].shape}")
        print(f"   Quats shape: {gaussians_all['quats'].shape}")
        print(f"   Scales shape: {gaussians_all['scales'].shape}")
        print(f"   Opacities shape: {gaussians_all['opacities'].shape}")
        print(f"   Colors shape: {gaussians_all['colors'].shape}")
        
        # 检查数据有效性
        print(f"\n3. Checking data validity...")
        for key_name, tensor in gaussians_all.items():
            if torch.isnan(tensor).any():
                print(f"   WARNING: NaN detected in {key_name}!")
            if torch.isinf(tensor).any():
                print(f"   WARNING: Inf detected in {key_name}!")
            print(f"   {key_name}: min={tensor.min().item():.4f}, max={tensor.max().item():.4f}, mean={tensor.mean().item():.4f}")
        
        print(f"\n4. Node states summary:")
        print(f"   node_state_bg: {node_state_bg.means.shape[0]} Gaussians")
        print(f"   node_state_rigid: {node_state_rigid.means.shape[0] if node_state_rigid is not None else 0} Gaussians")
        print(f"   node_state_distant: {node_state_distant.means.shape[0] if node_state_distant is not None else 0} Gaussians")
        
        print(f"\n✓ Gaussians preparation completed successfully!")
    else:
        print("Error: No batch available. Please create a batch first.")

Gaussians All Rendering Check Demo
Note: This demo shows all Gaussians (including invisible rigid points).
      For training, rigid points are filtered by visibility masks.
      See docs/analysis/rigid_visibility_filtering_minimal_change_proposal.md for details.

1. Getting node states from _get_or_init_node_states...
   Key: (1, 0)
   Background NodeState: 500000 Gaussians
   Rigid NodeState: None
   Distant NodeState: None
   No rigid node state, using default source_frame_idx: 0

2. Preparing all Gaussians using _prepare_all_gaussians (source_frame_idx=0)...
   Note: This includes ALL rigid points for visualization.
         In training, only visible rigid points are used (see visibility filtering proposal).
   Merged Gaussians:
     Total: 500000 Gaussians
     Background: 500000
     Rigid: 0
     Distant: 0
   Means shape: torch.Size([500000, 3])
   Quats shape: torch.Size([500000, 4])
   Scales shape: torch.Size([500000, 3])
   Opacities shape: torch.Size([500000])
   Colors s

In [16]:
scene_id, segment_id


(1, 0)

In [17]:
# 创建 nerfview 渲染器来可视化 gaussians_all
# 参考 models/trainers/base.py:716-788
# 
# 注意：此可视化显示所有 Gaussians（包括不可见的 rigid 点）
# 在实际训练中，rigid 点会根据可见性 mask 进行过滤
# 参见 docs/analysis/rigid_visibility_filtering_minimal_change_proposal.md

if 'gaussians_all' in locals() and gaussians_all is not None:
    print("Setting up nerfview renderer for gaussians_all...")
    print("=" * 80)
    print("Note: This viewer shows ALL Gaussians (including invisible rigid points).")
    print("      In training, only visible rigid points are rendered per frame.")
    
    # 获取渲染配置
    render_cfg = trainer.render_cfg if hasattr(trainer, 'render_cfg') else None
    
    # 创建渲染函数（参考 base.py 的 _viewer_render_fn）
    @torch.no_grad()
    def viewer_render_fn(camera_state: nerfview.CameraState, img_wh: Tuple[int, int]):
        """Render function for nerfview viewer using gaussians_all.
        
        Note: This renders all Gaussians including invisible rigid points.
        In training, rigid points are filtered by visibility masks per frame.
        """
        W, H = img_wh
        c2w = camera_state.c2w
        K = camera_state.get_K(img_wh)
        c2w = torch.from_numpy(c2w).float().to(trainer.device)
        K = torch.from_numpy(K).float().to(trainer.device)
        
        # 创建相机对象
        cam = dataclass_camera(
            camtoworlds=c2w,
            camtoworlds_gt=c2w,
            Ks=K,
            H=H,
            W=W
        )
        
        # 从 gaussians_all 获取参数
        means = gaussians_all['means']
        quats = gaussians_all['quats']
        scales = gaussians_all['scales']
        opacities = gaussians_all['opacities']
        colors_sh = gaussians_all['colors']  # [N, num_sh, 3]
        
        # 计算 viewdirs 用于球谐函数
        viewdirs = means.detach() - cam.camtoworlds[..., :3, 3]  # [N, 3]
        viewdirs = viewdirs / (viewdirs.norm(dim=-1, keepdim=True) + 1e-8)
        
        # 使用球谐函数计算 RGB（参考 models/nodes/rigid.py）
        from gsplat.cuda._wrapper import spherical_harmonics
        sh_degree = trainer.sh_degree if hasattr(trainer, 'sh_degree') else 0
        n = sh_degree  # 使用完整的 SH 度数
        rgbs = spherical_harmonics(n, viewdirs, colors_sh)
        rgbs = torch.clamp(rgbs + 0.5, 0.0, 1.0)  # [N, 3]
        
        # 创建高斯数据类
        gs = dataclass_gs(
            _means=means,
            _scales=scales,
            _quats=quats,
            _rgbs=rgbs,
            _opacities=opacities.unsqueeze(-1) if opacities.dim() == 1 else opacities,
            detach_keys=[],
            extras=None
        )
        
        # 渲染参数
        packed = render_cfg.packed if render_cfg and hasattr(render_cfg, 'packed') else False
        absgrad = render_cfg.absgrad if render_cfg and hasattr(render_cfg, 'absgrad') else False
        sparse_grad = render_cfg.sparse_grad if render_cfg and hasattr(render_cfg, 'sparse_grad') else False
        antialiased = render_cfg.antialiased if render_cfg and hasattr(render_cfg, 'antialiased') else False
        
        # 执行渲染
        # 注意：这里渲染所有点，包括不可见的 rigid 点
        # 在实际训练中，rigid 点会根据可见性 mask 进行过滤
        render_colors, _, _ = rasterization(
            means=gs.means,
            quats=gs.quats,
            scales=gs.scales,
            opacities=gs.opacities.squeeze(),
            colors=gs.rgbs,
            viewmats=torch.linalg.inv(cam.camtoworlds)[None, ...],  # [C, 4, 4]
            Ks=cam.Ks[None, ...],  # [C, 3, 3]
            width=cam.W,
            height=cam.H,
            packed=packed,
            absgrad=absgrad,
            sparse_grad=sparse_grad,
            rasterize_mode="antialiased" if antialiased else "classic",
            radius_clip=4.0,  # skip GSs that have small image radius (in pixels)
        )
        return render_colors[0].cpu().numpy()
    
    # 初始化 viewer
    print("\nInitializing nerfview viewer...")
    port = 8080
    server = viser.ViserServer(port=port, verbose=False)
    viewer = nerfview.Viewer(
        server=server,
        render_fn=viewer_render_fn,
        mode="training",
    )
    
    print(f"\n✓ Viewer initialized successfully!")
    print(f"  Viewer running at: http://localhost:{port}")
    print(f"  Press Ctrl+C to stop the viewer")
    print(f"\n  Note: The viewer displays ALL merged Gaussians (background + rigid + distant)")
    print(f"        Total Gaussians: {gaussians_all['means'].shape[0]}")
    print(f"\n  Training behavior:")
    print(f"    - Only visible rigid points in source frame are used for 3D feature volume")
    print(f"    - Only visible rigid points in each target frame are rendered")
    print(f"    - See rigid_visibility_filtering_minimal_change_proposal.md for details")
    
    # 保存 viewer 和 server 以便后续使用
    gaussians_all_viewer = viewer
    gaussians_all_server = server
    
else:
    print("Error: gaussians_all not available. Please run the previous cell first.")

Setting up nerfview renderer for gaussians_all...
Note: This viewer shows ALL Gaussians (including invisible rigid points).
      In training, only visible rigid points are rendered per frame.

Initializing nerfview viewer...


╭─────────────── viser ───────────────╮
│             ╷                       │
│   HTTP      │ http://0.0.0.0:8080   │
│   Websocket │ ws://0.0.0.0:8080     │
│             ╵                       │
╰─────────────────────────────────────╯


✓ Viewer initialized successfully!
  Viewer running at: http://localhost:8080
  Press Ctrl+C to stop the viewer

  Note: The viewer displays ALL merged Gaussians (background + rigid + distant)
        Total Gaussians: 500000

  Training behavior:
    - Only visible rigid points in source frame are used for 3D feature volume
    - Only visible rigid points in each target frame are rendered
    - See rigid_visibility_filtering_minimal_change_proposal.md for details


In [ ]:
# 绘制损失曲线
if len(losses) > 0:
    plt.figure(figsize=(10, 6))
    plt.plot(losses, marker='o', linestyle='-', linewidth=2, markersize=8)
    plt.xlabel('Iteration', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('Training Loss Curve', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"\nLoss statistics:")
    print(f"  Iterations: {len(losses)}")
    print(f"  Final loss: {losses[-1]:.6f}")
    print(f"  Average loss: {np.mean(losses):.6f}")
    print(f"  Std loss: {np.std(losses):.6f}")
else:
    print("No loss data available. Please run the training loop first.")

No loss data available. Please run the training loop first.
